# Snowflake Crime Data ETL Pipeline Project

## Introduction

This project was developed to support operational crime monitoring and reporting across UK police forces. The objective of this project was to design and implement a scalable Python-based, Snowflake data ETL pipeline that transforms raw crime data into a clean, aggregated, and reporting-ready dataset suitable for use in BI analysis. 

The pipleline processes UK Police crime data for four selected police forces and integrates multiple enrichment datasets. These enrichment datasets allow for more meaningful comparisons between police forces across demographic and socioeconomic factors. 

To support scalable and reliable reporting, the pipeline was structured into distinct engineering stages including: 

• Batch ingestion of raw crime data  
• Enrichment dataset integration  
• Data cleaning and validation  
• Feature engineering and transformation  
• Aggregation to final reporting grain  
• and Export of final BI-ready reporting dataset 

Particular emphasis was placed on data quality and validation throughout the pipeline. Validation checks were implemented at every stage to ensure each stage worked as intended. Reporting-grain validation was also conducted to ensure the final dataset maintained a single, consistent reporting grain. 

The final dataset was aggregated to reporting grain of: 

***Force x Year x Quater x District x Crime Type***

This grain was chosen to support time-series analysis, force-level and district comparisons, and crime category monitoring. 

## Workflow Diagram 

*insert workflow diagram

## Data Assumptions and Limitations

- Input column names match the public data.police.uk street-crime extract.
- CRIME_ID is the record-level uniqueness key; this should be confirmed with the project's ingestion owner.
- Missing outcomes and location descriptions are retained because they are not essential to the intended reporting grain.
- LSOA and coordinate completeness are enforced to match the original notebook's cleaning policy and support later geographic analysis.
- Coordinate checks use broad UK bounds, not a force-boundary spatial check.
- New Police.uk crime categories will deliberately fail validation until reviewed and added to the approved category set.

## Import Python Libraries 

In [ ]:
#data handling libraries 
import pandas as pd 
import numpy as np

#any other libraries go 
from datetime import datetime, timezone
import re
import uuid

## Ingestion Layer - edit!

**Outline:** 

This layer follows the steps taken to extract crime and police force data from data.police.uk, as well as ONS deprivation and population data for later cleaning and transformation. 

The following ingestion tasks were completed: 

- Created an AWS account and shared S3 bucket used by the group.
- Created an IAM user and permissions policy that allowed the local Python notebooks to upload files to the required locations in the bucket.
- Created a shared AWS IAM role that allows the team’s separate Snowflake accounts to access the S3 bucket.
- Organised the bucket into separate prefixes for raw crime data, cleaned crime data, population data, English deprivation data and Welsh deprivation data.
- Wrote Python notebooks containing functions that download each dataset and upload them to the appropriate S3 prefixes.
- Created and tested Snowflake storage integration, file formats, and external stages to confirm that the files could be accessed from the bucket.

*As Snowflake trial account usage is limited, the ingestion of crime and police force data from data.police.uk has been completed in Jupyter Notebook. 

Refer to the following files for the complete data ingestion process: 

- Crime Ingestion: /crime_ingestion.ipynb
- Population Ingestion: /pop_ingestion.ipynb
- Deprivation Ingestion: /dep_ingestion.ipynb

### Step 1: Create Storage Integration 

A storage integration was created to store raw & clean crime, population, and deprivation datasets. 

Purpose:  
 
- Connect Snowflake to the raw and clean S3 locations.  

Important:
- This script does not alter or remove files from the raw S3 location.
- CREATE and COPY operations only create Snowflake objects and read the source files.
- The storage integration may already exist. If it does, ALTER is used to retain the complete list of approved S3 locations.

In [ ]:
%%sql -r dataframe_1
/* Account-level objects and permissions require ACCOUNTADMIN. */
USE ROLE ACCOUNTADMIN;

CREATE STORAGE INTEGRATION IF NOT EXISTS CRIME_S3_INTEGRATION
    TYPE = EXTERNAL_STAGE
    STORAGE_PROVIDER = 'S3'
    ENABLED = TRUE
    STORAGE_AWS_ROLE_ARN =
        'arn:aws:iam::559852958324:role/SnowflakeCrimeS3Role'
    STORAGE_ALLOWED_LOCATIONS = (
        's3://rockborne-ch19-g1-crime/raw/uk-police/',
        's3://rockborne-ch19-g1-crime/clean/uk-police/',
        's3://rockborne-ch19-g1-crime/raw/enrichment/population/',
        's3://rockborne-ch19-g1-crime/raw/enrichment/deprivation/england/',
        's3://rockborne-ch19-g1-crime/raw/enrichment/deprivation/wales/'
    );

DESC INTEGRATION CRIME_S3_INTEGRATION;

GRANT USAGE ON INTEGRATION CRIME_S3_INTEGRATION TO ROLE SYSADMIN;

** Access to the s3 bucket is dependent on providing the Ingestion Engineer with the `STORAGE_AWS_IAM_USER_ARN` and `STORAGE_AWS_EXTERNAL_ID` property values, produced by the `DESC INTEGRATION CRIME_S3_INTEGRATION;` and account permissions being granted. 

### Step 2: Create Crime Database

Purpose:  
 
- Create the database objects required by the cleaning & validation layer.  
- Load the raw street-crime CSV files into CRIME_ETL_DB.RAW.STREET_CRIME.

In [ ]:
%%sql -r dataframe_2
/* Create the project database and allow SYSADMIN to create its schemas. */
CREATE DATABASE IF NOT EXISTS CRIME_ETL_DB;
GRANT USAGE ON DATABASE CRIME_ETL_DB TO ROLE SYSADMIN;
GRANT CREATE SCHEMA ON DATABASE CRIME_ETL_DB TO ROLE SYSADMIN;

/* All remaining project objects are owned or operated by SYSADMIN. */
USE ROLE SYSADMIN;
USE DATABASE CRIME_ETL_DB;

CREATE SCHEMA IF NOT EXISTS RAW;
CREATE SCHEMA IF NOT EXISTS DATA_QUALITY;
CREATE SCHEMA IF NOT EXISTS CLEAN;

In [ ]:
%%sql -r dataframe_3
USE DATABASE CRIME_ETL_DB;
USE SCHEMA RAW;

/*
    Police.uk files have one header row.
*/
CREATE OR REPLACE FILE FORMAT CRIME_CSV_FORMAT
    TYPE = CSV
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    EMPTY_FIELD_AS_NULL = TRUE
    TRIM_SPACE = TRUE
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE;

/*
    External stage pointing to the raw Police.uk prefix.
*/
CREATE OR REPLACE STAGE CRIME_S3_STAGE
    URL = 's3://rockborne-ch19-g1-crime/raw/uk-police/'
    STORAGE_INTEGRATION = CRIME_S3_INTEGRATION
    FILE_FORMAT = CRIME_CSV_FORMAT;

/*
    Confirm that the stage was created.
*/
--SHOW STAGES LIKE 'CRIME_S3_STAGE'
    --IN SCHEMA CRIME_ETL_DB.RAW;

/*
    Confirm that Snowflake can access the S3 files.
*/
LIST @CRIME_ETL_DB.RAW.CRIME_S3_STAGE;

In [ ]:
%%sql -r dataframe_4
/*
    Raw landing table expected by Police_Crime_Snowflake_Cleaning_Validation.ipynb.
    Text is retained for most source fields so cleaning and type validation happen
    in the cleaning layer rather than silently changing the source values on load.
*/

DROP TABLE CRIME_ETL_DB.RAW.STREET_CRIME; 

CREATE TABLE IF NOT EXISTS CRIME_ETL_DB.RAW.STREET_CRIME (
    "Crime ID" TEXT,
    "Month" TEXT,
    "Reported by" TEXT,
    "Falls within" TEXT,
    "Longitude" TEXT,
    "Latitude" TEXT,
    "Location" TEXT,
    "LSOA code" TEXT,
    "LSOA name" TEXT,
    "Crime type" TEXT,
    "Last outcome category" TEXT,
    "Context" TEXT
);

/*
    Load only the four police forces included in this project.
    PATTERN searches file paths and names case-insensitively.
    FORCE = FALSE avoids reloading files already recorded in Snowflake load history.
*/
COPY INTO CRIME_ETL_DB.RAW.STREET_CRIME
FROM @CRIME_ETL_DB.RAW.CRIME_S3_STAGE
FILE_FORMAT = (FORMAT_NAME = CRIME_ETL_DB.RAW.CRIME_CSV_FORMAT)
PATTERN = '.*(metropolitan|west-midlands|south-wales|sussex).*street\\.csv'
ON_ERROR = 'CONTINUE'
FORCE = FALSE;

In [ ]:
%%sql -r dataframe_5
/* Basic checks after the load. */
SELECT COUNT(*) AS RAW_ROWS
FROM CRIME_ETL_DB.RAW.STREET_CRIME;

SELECT
    "Falls within" AS FORCE_NAME,
    COUNT(*) AS RAW_ROWS
FROM CRIME_ETL_DB.RAW.STREET_CRIME
GROUP BY "Falls within"
ORDER BY FORCE_NAME;

In [ ]:
%%sql -r dataframe_6
/*
    Shared clean external stage used by both crime & supplementry data cleaning steps for their CSV exports.
    Creating a stage does not write anything to S3; COPY INTO the stage performs
    the export later.
*/
USE SCHEMA CRIME_ETL_DB.CLEAN;

CREATE STAGE IF NOT EXISTS CRIME_CLEAN_STAGE
    URL = 's3://rockborne-ch19-g1-crime/clean/uk-police/'
    STORAGE_INTEGRATION = CRIME_S3_INTEGRATION;

LIST @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE;

### Step 3: Create Supplementary Data File Formats & Schemas

Datasets:
- Police-force-area population data
- England Index of Multiple Deprivation (IMD)
 - Wales Index of Multiple Deprivation (WIMD)

The supplementary cleaning step reads these external stages directly. It does not require separate raw Snowflake tables.

Important:
- The raw S3 files remain untouched.
- This script only creates file formats and references to the S3 locations.
- The source files must be CSV files before the notebook is run.

In [ ]:
%%sql -r dataframe_7
/*
    Keep the header rows available to the notebook. Its parsing logic explicitly
    identifies and excludes headers while retaining source-row visibility.
*/ 
USE SCHEMA RAW;

CREATE FILE FORMAT IF NOT EXISTS POPULATION_CSV_FORMAT
    TYPE = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    EMPTY_FIELD_AS_NULL = TRUE
    TRIM_SPACE = TRUE
    SKIP_HEADER = 0
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE;

/*
    Population stage expected by the notebook.
    The original workbook must be exported to CSV and placed in this /csv/ folder.
*/
CREATE STAGE IF NOT EXISTS POPULATION_S3_STAGE
    URL = 's3://rockborne-ch19-g1-crime/raw/enrichment/population/csv/'
    STORAGE_INTEGRATION = CRIME_S3_INTEGRATION
    FILE_FORMAT = POPULATION_CSV_FORMAT;

In [ ]:
%%sql -r dataframe_9
USE SCHEMA RAW;

CREATE FILE FORMAT IF NOT EXISTS DEPRIVATION_ENGLAND_CSV_FORMAT
    TYPE = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    EMPTY_FIELD_AS_NULL = TRUE
    TRIM_SPACE = TRUE
    SKIP_HEADER = 0
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE;

/* England IMD stage expected by the notebook. */
CREATE STAGE IF NOT EXISTS DEPRIVATION_ENGLAND_S3_STAGE
    URL = 's3://rockborne-ch19-g1-crime/raw/enrichment/deprivation/england/'
    STORAGE_INTEGRATION = CRIME_S3_INTEGRATION
    FILE_FORMAT = DEPRIVATION_ENGLAND_CSV_FORMAT;

In [ ]:
%%sql -r dataframe_10
USE SCHEMA RAW;

CREATE FILE FORMAT IF NOT EXISTS DEPRIVATION_WALES_CSV_FORMAT
    TYPE = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    EMPTY_FIELD_AS_NULL = TRUE
    TRIM_SPACE = TRUE
    SKIP_HEADER = 0
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE;

/* Wales WIMD stage expected by the notebook. */
CREATE STAGE IF NOT EXISTS DEPRIVATION_WALES_S3_STAGE
    URL = 's3://rockborne-ch19-g1-crime/raw/enrichment/deprivation/wales/'
    STORAGE_INTEGRATION = CRIME_S3_INTEGRATION
    FILE_FORMAT = DEPRIVATION_WALES_CSV_FORMAT;

In [ ]:
%%sql -r dataframe_11
/* Confirm that Snowflake can see the files in each source location. */
LIST @CRIME_ETL_DB.RAW.POPULATION_S3_STAGE;
LIST @CRIME_ETL_DB.RAW.DEPRIVATION_ENGLAND_S3_STAGE;
LIST @CRIME_ETL_DB.RAW.DEPRIVATION_WALES_S3_STAGE;

In [ ]:
%%sql -r dataframe_12
/*
    Preview the population source. The notebook expects:
    $1 = police-force-area code
    $2 = police-force-area name
    $3 = year
    $4 to $175 = female and male population counts by age.
*/
SELECT
    METADATA$FILENAME AS FILE_NAME,
    METADATA$FILE_ROW_NUMBER AS ROW_NUMBER,
    t.$1 AS PFA_CODE,
    t.$2 AS FORCE_NAME,
    t.$3 AS YEAR,
    t.$4 AS FIRST_POPULATION_FIELD,
    t.$175 AS LAST_POPULATION_FIELD
FROM @CRIME_ETL_DB.RAW.POPULATION_S3_STAGE t
LIMIT 30;

In [ ]:
%%sql -r dataframe_13
/*
    Preview the England IMD source.
    The notebook uses 56 columns, beginning with the 2021 LSOA fields.
*/
SELECT
    METADATA$FILENAME AS FILE_NAME,
    METADATA$FILE_ROW_NUMBER AS ROW_NUMBER,
    t.$1 AS LSOA_CODE,
    t.$2 AS LSOA_NAME,
    t.$3 AS LOCAL_AUTHORITY_CODE,
    t.$4 AS LOCAL_AUTHORITY_NAME,
    t.$5 AS IMD_SCORE,
    t.$6 AS IMD_RANK,
    t.$7 AS IMD_DECILE,
    t.$56 AS WORKING_AGE_POPULATION
FROM @CRIME_ETL_DB.RAW.DEPRIVATION_ENGLAND_S3_STAGE t
LIMIT 30;


In [ ]:
%%sql -r dataframe_14
/*
    Preview the Wales WIMD long-format source.
    The notebook uses the six analytical fields shown below and ignores the
    reference, sorting and hierarchy metadata fields.
*/
SELECT
    METADATA$FILENAME AS FILE_NAME,
    METADATA$FILE_ROW_NUMBER AS ROW_NUMBER,
    t.$1 AS DATA_VALUE,
    t.$3 AS DATA_DESCRIPTION,
    t.$7 AS AREA_CODE,
    t.$11 AS AREA_NAME,
    t.$15 AS DOMAIN,
    t.$19 AS NOTES
FROM @CRIME_ETL_DB.RAW.DEPRIVATION_WALES_S3_STAGE t
LIMIT 30;

## Cleaning & Validation Layer 

### Step 1: Police Crime Cleaning and Validation

#### Outline:

Cleaning and validation only for Metropolitan Police, West Midlands Police, South Wales Police, and Sussex Police.

This step begins with a raw Snowflake table supplied by the ingestion team and produces:

- a standardised, deduplicated clean table for downstream teammates;
- a quarantine table containing every rejected row and its rejection reason;
- run-level and force-level audit metrics;
- an explicit validation-results table.

It deliberately excludes enrichment, aggregation, trend analysis, visualisation, and Power BI export.

#### Ownership boundary

**Input contract:** one raw table containing the standard data.police.uk street-crime columns. The raw table is not changed.

**Output contract:** record-level clean and quarantine tables. A downstream teammate may enrich and aggregate the clean table.

#### Cleaning Rules

1. Trim text and turn blank strings into `NULL`.
2. Standardise the four force names and a small set of known crime-category aliases.
3. Parse `MONTH` as a first-of-month date; parse coordinates with `TRY_TO_DOUBLE`.
4. Reject rows with missing/invalid critical fields, invalid coordinates, unknown force/category, or missing location identifiers.
5. Deduplicate by `CRIME_ID`, retaining the most complete row deterministically.
6. Preserve all rejected rows in quarantine.

The original notebook dropped duplicate IDs with `keep=False`. Here they are handled more conservatively: one canonical row is kept and extra copies are quarantined.

In [ ]:
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception as exc:
    raise RuntimeError(
        "Run this notebook inside Snowflake, or replace this block with a configured "
        "Snowpark Session. No credentials should be written into the notebook."
    ) from exc

RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)

# Edit only these object names. Identifiers are validated before use.
RAW_TABLE = "CRIME_ETL_DB.RAW.STREET_CRIME"
WORK_SCHEMA = "CRIME_ETL_DB.DATA_QUALITY"
CLEAN_TABLE = f"{WORK_SCHEMA}.STREET_CRIME_CLEAN"
QUARANTINE_TABLE = f"{WORK_SCHEMA}.STREET_CRIME_QUARANTINE"
RUN_AUDIT_TABLE = f"{WORK_SCHEMA}.CLEANING_RUN_AUDIT"
FORCE_AUDIT_TABLE = f"{WORK_SCHEMA}.CLEANING_FORCE_AUDIT"
VALIDATION_TABLE = f"{WORK_SCHEMA}.VALIDATION_RESULTS"

IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*(\.[A-Za-z_][A-Za-z0-9_]*){0,2}$")
for name in [
    RAW_TABLE, WORK_SCHEMA, CLEAN_TABLE, QUARANTINE_TABLE,
    RUN_AUDIT_TABLE, FORCE_AUDIT_TABLE, VALIDATION_TABLE
]:
    if not IDENTIFIER.fullmatch(name):
        raise ValueError(f"Unsafe Snowflake object identifier: {name}")

print(f"Run ID: {RUN_ID}")
print(f"Source: {RAW_TABLE}")

## 1. Confirm the input contract

The source table must contain these columns (case-insensitive). `SOURCE_FILE_NAME` is recommended for lineage but is not required.

In [ ]:

REQUIRED_COLUMNS = {
    "CRIME ID", "MONTH", "REPORTED BY", "FALLS WITHIN", "LONGITUDE", "LATITUDE",
    "LOCATION", "LSOA CODE", "LSOA NAME", "CRIME TYPE", "LAST OUTCOME CATEGORY"
}

def show_column_name(row):
    values = row.as_dict()
    for key, value in values.items():
        if key.lower() in {"column_name", "name"}:
            return str(value)
    raise KeyError(f"Could not find column-name field in SHOW COLUMNS output: {values.keys()}")

source_columns = {
    show_column_name(r).upper()
    for r in session.sql(f"SHOW COLUMNS IN TABLE {RAW_TABLE}").collect()
}
missing_columns = REQUIRED_COLUMNS - source_columns
if missing_columns:
    raise ValueError(f"Source contract failed. Missing columns: {sorted(missing_columns)}")

HAS_SOURCE_FILE = "SOURCE_FILE_NAME" in source_columns
print("PASS - source contract satisfied")
print(f"SOURCE_FILE_NAME available: {HAS_SOURCE_FILE}")

## 2. Create audit structures


In [ ]:
session.sql(f"CREATE SCHEMA IF NOT EXISTS {WORK_SCHEMA}").collect()

session.sql(f"""
CREATE TABLE IF NOT EXISTS {RUN_AUDIT_TABLE} (
    RUN_ID STRING, STARTED_AT TIMESTAMP_TZ, COMPLETED_AT TIMESTAMP_TZ,
    SOURCE_TABLE STRING, CLEAN_TABLE STRING, QUARANTINE_TABLE STRING,
    RAW_TARGET_ROWS NUMBER, CLEAN_ROWS NUMBER, QUARANTINE_ROWS NUMBER,
    ROWS_REMOVED NUMBER, REMOVAL_PERCENT NUMBER(10,4), STATUS STRING
)
""").collect()

session.sql(f"""
CREATE TABLE IF NOT EXISTS {FORCE_AUDIT_TABLE} (
    RUN_ID STRING, FORCE_NAME STRING, RAW_ROWS NUMBER, CLEAN_ROWS NUMBER,
    QUARANTINE_ROWS NUMBER, DUPLICATE_ROWS NUMBER, NULL_CRIME_ID_ROWS NUMBER,
    NULL_MONTH_ROWS NUMBER, NULL_LSOA_ROWS NUMBER, NULL_COORDINATE_ROWS NUMBER
)
""").collect()

session.sql(f"""
CREATE TABLE IF NOT EXISTS {VALIDATION_TABLE} (
    RUN_ID STRING, CHECK_NAME STRING, SEVERITY STRING, OBSERVED_VALUE STRING,
    EXPECTED_VALUE STRING, PASSED BOOLEAN, CHECKED_AT TIMESTAMP_TZ
)
""").collect()
print("Audit structures ready")

## 3. Standardise and assess quality

Only the four in-scope forces are selected. The temporary stage exists only for this Snowflake session and does not modify the raw data.

In [ ]:
source_file_expression = (
    'NULLIF(TRIM("SOURCE_FILE_NAME"), \'\')'
    if HAS_SOURCE_FILE else "CAST(NULL AS STRING)"
)

session.sql(f"""
CREATE OR REPLACE TEMPORARY TABLE STG_POLICE_CRIME_STANDARDISED AS
WITH NORMALISED AS (
    SELECT
        NULLIF(TRIM("Crime ID"), '') AS CRIME_ID,
        TRY_TO_DATE(NULLIF(TRIM("Month"), '') || '-01') AS CRIME_MONTH,
        NULLIF(TRIM("Reported by"), '') AS REPORTED_BY_RAW,
        NULLIF(TRIM("Falls within"), '') AS FALLS_WITHIN_RAW,
        TRY_TO_DOUBLE("Longitude") AS LONGITUDE,
        TRY_TO_DOUBLE("Latitude") AS LATITUDE,
        NULLIF(TRIM("Location"), '') AS LOCATION,
        UPPER(NULLIF(TRIM("LSOA code"), '')) AS LSOA_CODE,
        NULLIF(TRIM("LSOA name"), '') AS LSOA_NAME,
        NULLIF(TRIM("Crime type"), '') AS CRIME_TYPE_RAW,
        NULLIF(TRIM("Last outcome category"), '') AS LAST_OUTCOME_CATEGORY,
        {source_file_expression} AS SOURCE_FILE_NAME,
        CURRENT_TIMESTAMP() AS CLEANED_AT,
        '{RUN_ID}' AS CLEANING_RUN_ID
    FROM {RAW_TABLE}
    WHERE LOWER(TRIM("Falls within")) IN (
        'metropolitan police', 'metropolitan police service',
        'west midlands police', 'south wales police', 'sussex police'
    )
), STANDARDISED AS (
    SELECT *,
        CASE
            WHEN LOWER(FALLS_WITHIN_RAW) IN
                ('metropolitan police', 'metropolitan police service')
                THEN 'Metropolitan Police Service'
            WHEN LOWER(FALLS_WITHIN_RAW) = 'west midlands police'
                THEN 'West Midlands Police'
            WHEN LOWER(FALLS_WITHIN_RAW) = 'south wales police'
                THEN 'South Wales Police'
            WHEN LOWER(FALLS_WITHIN_RAW) = 'sussex police'
                THEN 'Sussex Police'
        END AS FORCE_NAME,
        CASE LOWER(CRIME_TYPE_RAW)
            WHEN 'anti social behaviour' THEN 'Anti-social behaviour'
            WHEN 'antisocial behaviour' THEN 'Anti-social behaviour'
            WHEN 'anti-social behaviour' THEN 'Anti-social behaviour'
            WHEN 'bicycle theft' THEN 'Bicycle theft'
            WHEN 'burglary' THEN 'Burglary'
            WHEN 'criminal damage and arson' THEN 'Criminal damage and arson'
            WHEN 'drugs' THEN 'Drugs'
            WHEN 'other crime' THEN 'Other crime'
            WHEN 'other theft' THEN 'Other theft'
            WHEN 'possession of weapons' THEN 'Possession of weapons'
            WHEN 'public order' THEN 'Public order'
            WHEN 'robbery' THEN 'Robbery'
            WHEN 'shoplifting' THEN 'Shoplifting'
            WHEN 'theft from the person' THEN 'Theft from the person'
            WHEN 'vehicle crime' THEN 'Vehicle crime'
            WHEN 'violence & sexual offences' THEN 'Violence and sexual offences'
            WHEN 'violence and sexual offences' THEN 'Violence and sexual offences'
            ELSE CRIME_TYPE_RAW
        END AS CRIME_TYPE
    FROM NORMALISED
), ASSESSED AS (
    SELECT *,
        ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
            IFF(CRIME_ID IS NULL, 'MISSING_CRIME_ID', NULL),
            IFF(CRIME_MONTH IS NULL, 'INVALID_OR_MISSING_MONTH', NULL),
            IFF(FORCE_NAME IS NULL, 'UNKNOWN_FORCE', NULL),
            IFF(CRIME_TYPE IS NULL, 'MISSING_CRIME_TYPE', NULL),
            IFF(LSOA_CODE IS NULL, 'MISSING_LSOA_CODE', NULL),
            IFF(LATITUDE IS NULL OR LONGITUDE IS NULL, 'MISSING_COORDINATE', NULL),
            IFF(LATITUDE IS NOT NULL AND NOT (LATITUDE BETWEEN 49 AND 61),
                'LATITUDE_OUT_OF_UK_RANGE', NULL),
            IFF(LONGITUDE IS NOT NULL AND NOT (LONGITUDE BETWEEN -9 AND 3),
                'LONGITUDE_OUT_OF_UK_RANGE', NULL)
        ), '|') AS BASE_REJECTION_REASON,
        ROW_NUMBER() OVER (
            PARTITION BY CRIME_ID
            ORDER BY
                IFF(CRIME_MONTH IS NOT NULL, 1, 0)
                + IFF(LSOA_CODE IS NOT NULL, 1, 0)
                + IFF(LATITUDE IS NOT NULL, 1, 0)
                + IFF(LONGITUDE IS NOT NULL, 1, 0) DESC,
                SOURCE_FILE_NAME NULLS LAST, FORCE_NAME
        ) AS CRIME_ID_RANK
    FROM STANDARDISED
)
SELECT *,
    CASE
        WHEN BASE_REJECTION_REASON <> '' AND CRIME_ID_RANK > 1
            THEN BASE_REJECTION_REASON || '|DUPLICATE_CRIME_ID'
        WHEN BASE_REJECTION_REASON <> '' THEN BASE_REJECTION_REASON
        WHEN CRIME_ID_RANK > 1 THEN 'DUPLICATE_CRIME_ID'
        ELSE NULL
    END AS REJECTION_REASON
FROM ASSESSED
""").collect()

raw_target_rows = session.table("STG_POLICE_CRIME_STANDARDISED").count()
if raw_target_rows == 0:
    raise ValueError("No rows found for the four target forces; check source table and names.")
print(f"Rows assessed: {raw_target_rows:,}")

## 4. Publish clean and quarantine tables

The SWAP WITH pattern replaces each published table atomically after a complete candidate table has been built. Consumers therefore do not see a half-written result.

In [ ]:
clean_projection = """
CRIME_ID, CRIME_MONTH, YEAR(CRIME_MONTH) AS CRIME_YEAR,
MONTH(CRIME_MONTH) AS CRIME_MONTH_NUMBER, REPORTED_BY_RAW AS REPORTED_BY,
FORCE_NAME, LONGITUDE, LATITUDE, LOCATION, LSOA_CODE, LSOA_NAME, CRIME_TYPE,
LAST_OUTCOME_CATEGORY, SOURCE_FILE_NAME, CLEANED_AT, CLEANING_RUN_ID
"""

session.sql(f"""
CREATE OR REPLACE TABLE {CLEAN_TABLE}__CANDIDATE AS
SELECT {clean_projection}
FROM STG_POLICE_CRIME_STANDARDISED
WHERE REJECTION_REASON IS NULL
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {QUARANTINE_TABLE}__CANDIDATE AS
SELECT {clean_projection}, REJECTION_REASON
FROM STG_POLICE_CRIME_STANDARDISED
WHERE REJECTION_REASON IS NOT NULL
""").collect()

def publish_candidate(target):
    exists = session.sql(
        f"SHOW TABLES LIKE '{target.split('.')[-1]}' IN SCHEMA {WORK_SCHEMA}"
    ).count() > 0
    if exists:
        session.sql(f"ALTER TABLE {target} SWAP WITH {target}__CANDIDATE").collect()
        session.sql(f"DROP TABLE {target}__CANDIDATE").collect()
    else:
        session.sql(f"ALTER TABLE {target}__CANDIDATE RENAME TO {target}").collect()

publish_candidate(CLEAN_TABLE)
publish_candidate(QUARANTINE_TABLE)
print("Clean and quarantine tables published")
     

## 5. Record cleaning metrics

Counts are reconciled at run and force level. Duplicate counts represent extra copies quarantined after deterministic ranking.

In [ ]:
	

session.sql(f"""
INSERT INTO {FORCE_AUDIT_TABLE}
SELECT
    '{RUN_ID}', FORCE_NAME,
    COUNT(*) AS RAW_ROWS,
    COUNT_IF(REJECTION_REASON IS NULL) AS CLEAN_ROWS,
    COUNT_IF(REJECTION_REASON IS NOT NULL) AS QUARANTINE_ROWS,
    COUNT_IF(CRIME_ID_RANK > 1) AS DUPLICATE_ROWS,
    COUNT_IF(CRIME_ID IS NULL) AS NULL_CRIME_ID_ROWS,
    COUNT_IF(CRIME_MONTH IS NULL) AS NULL_MONTH_ROWS,
    COUNT_IF(LSOA_CODE IS NULL) AS NULL_LSOA_ROWS,
    COUNT_IF(LATITUDE IS NULL OR LONGITUDE IS NULL) AS NULL_COORDINATE_ROWS
FROM STG_POLICE_CRIME_STANDARDISED
GROUP BY FORCE_NAME
""").collect()

clean_rows = session.table(CLEAN_TABLE).count()
quarantine_rows = session.table(QUARANTINE_TABLE).count()
removal_pct = round(quarantine_rows / raw_target_rows * 100, 4)

session.sql(f"""
INSERT INTO {RUN_AUDIT_TABLE}
SELECT '{RUN_ID}', TO_TIMESTAMP_TZ('{RUN_STARTED_AT.isoformat()}'),
       CURRENT_TIMESTAMP(), '{RAW_TABLE}', '{CLEAN_TABLE}', '{QUARANTINE_TABLE}',
       {raw_target_rows}, {clean_rows}, {quarantine_rows}, {quarantine_rows},
       {removal_pct}, 'CLEANED_PENDING_VALIDATION'
""").collect()

session.sql(f"""
SELECT FORCE_NAME, RAW_ROWS, CLEAN_ROWS, QUARANTINE_ROWS, DUPLICATE_ROWS,
       NULL_CRIME_ID_ROWS, NULL_MONTH_ROWS, NULL_LSOA_ROWS, NULL_COORDINATE_ROWS
FROM {FORCE_AUDIT_TABLE}
WHERE RUN_ID = '{RUN_ID}'
ORDER BY FORCE_NAME
""").show()

## 6. Explicit validation suite

Critical failures stop the handoff. Warnings are recorded but do not fail the run. Every check is written to Snowflake for auditability.

In [ ]:
EXPECTED_FORCES = {
    "Metropolitan Police Service", "West Midlands Police",
    "South Wales Police", "Sussex Police"
}

checks = []

def add_check(name, severity, observed, expected, passed):
    checks.append({
        "check_name": name, "severity": severity,
        "observed": str(observed), "expected": str(expected), "passed": bool(passed)
    })

add_check(
    "row_count_reconciliation", "CRITICAL",
    clean_rows + quarantine_rows, raw_target_rows,
    clean_rows + quarantine_rows == raw_target_rows
)
add_check("clean_table_not_empty", "CRITICAL", clean_rows, "> 0", clean_rows > 0)

actual_forces = {
    r["FORCE_NAME"]
    for r in session.sql(f"SELECT DISTINCT FORCE_NAME FROM {CLEAN_TABLE}").collect()
}
add_check(
    "four_force_coverage", "CRITICAL", sorted(actual_forces),
    sorted(EXPECTED_FORCES), actual_forces == EXPECTED_FORCES
)

critical_nulls = session.sql(f"""
SELECT COUNT(*) AS N FROM {CLEAN_TABLE}
WHERE CRIME_ID IS NULL OR CRIME_MONTH IS NULL OR FORCE_NAME IS NULL
   OR CRIME_TYPE IS NULL OR LSOA_CODE IS NULL
   OR LATITUDE IS NULL OR LONGITUDE IS NULL
""").collect()[0]["N"]
add_check("required_fields_complete", "CRITICAL", critical_nulls, 0, critical_nulls == 0)

duplicate_ids = session.sql(f"""
SELECT COUNT(*) AS N FROM (
    SELECT CRIME_ID FROM {CLEAN_TABLE}
    GROUP BY CRIME_ID HAVING COUNT(*) > 1
)
""").collect()[0]["N"]
add_check("crime_id_unique", "CRITICAL", duplicate_ids, 0, duplicate_ids == 0)

invalid_coordinates = session.sql(f"""
SELECT COUNT(*) AS N FROM {CLEAN_TABLE}
WHERE LATITUDE NOT BETWEEN 49 AND 61 OR LONGITUDE NOT BETWEEN -9 AND 3
""").collect()[0]["N"]
add_check("uk_coordinate_range", "CRITICAL", invalid_coordinates, 0, invalid_coordinates == 0)

future_months = session.sql(f"""
SELECT COUNT(*) AS N FROM {CLEAN_TABLE}
WHERE CRIME_MONTH > DATE_TRUNC('MONTH', CURRENT_DATE())
""").collect()[0]["N"]
add_check("no_future_months", "WARNING", future_months, 0, future_months == 0)

allowed_categories = {
    "Anti-social behaviour", "Bicycle theft", "Burglary", "Criminal damage and arson",
    "Drugs", "Other crime", "Other theft", "Possession of weapons",
    "Public order", "Robbery", "Shoplifting", "Theft from the person",
    "Vehicle crime", "Violence and sexual offences"
}
actual_categories = {
    r["CRIME_TYPE"]
    for r in session.sql(f"SELECT DISTINCT CRIME_TYPE FROM {CLEAN_TABLE}").collect()
}
unknown_categories = sorted(actual_categories - allowed_categories)
add_check(
    "recognised_crime_categories", "CRITICAL",
    unknown_categories, "[]", len(unknown_categories) == 0
)

for check in checks:
    observed = check["observed"].replace("'", "''")
    expected = check["expected"].replace("'", "''")
    session.sql(f"""
    INSERT INTO {VALIDATION_TABLE}
    SELECT '{RUN_ID}', '{check["check_name"]}', '{check["severity"]}',
           '{observed}', '{expected}', {str(check["passed"]).upper()}, CURRENT_TIMESTAMP()
    """).collect()

session.sql(f"""
SELECT CHECK_NAME, SEVERITY, OBSERVED_VALUE, EXPECTED_VALUE, PASSED
FROM {VALIDATION_TABLE}
WHERE RUN_ID = '{RUN_ID}'
ORDER BY IFF(SEVERITY = 'CRITICAL', 0, 1), CHECK_NAME
""").show()

critical_failures = [c for c in checks if c["severity"] == "CRITICAL" and not c["passed"]]
final_status = "VALIDATED" if not critical_failures else "VALIDATION_FAILED"
session.sql(f"""
UPDATE {RUN_AUDIT_TABLE}
SET STATUS = '{final_status}', COMPLETED_AT = CURRENT_TIMESTAMP()
WHERE RUN_ID = '{RUN_ID}'
""").collect()

if critical_failures:
    raise AssertionError(
        "Critical validation failure(s): "
        + ", ".join(c["check_name"] for c in critical_failures)
    )
print("PASS - all critical validations succeeded")

## 7. Handoff summary

Only rows from a `VALIDATED` run should be used downstream. The clean table remains at individual-crime grain; aggregation belongs to the downstream transformation owner.

In [ ]:

session.sql(f"""
SELECT RUN_ID, STARTED_AT, COMPLETED_AT, SOURCE_TABLE, CLEAN_TABLE,
       QUARANTINE_TABLE, RAW_TARGET_ROWS, CLEAN_ROWS, QUARANTINE_ROWS,
       REMOVAL_PERCENT, STATUS
FROM {RUN_AUDIT_TABLE}
WHERE RUN_ID = '{RUN_ID}'
""").show()

session.sql(f"""
SELECT REJECTION_REASON, COUNT(*) AS ROW_COUNT
FROM {QUARANTINE_TABLE}
GROUP BY REJECTION_REASON
ORDER BY ROW_COUNT DESC
""").show()

In [ ]:
%%sql -r dataframe_8
COPY INTO @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/metropolitan_police_clean.csv
FROM (
    SELECT *
    FROM CRIME_ETL_DB.DATA_QUALITY.STREET_CRIME_CLEAN
    WHERE FORCE_NAME = 'Metropolitan Police Service'
)
FILE_FORMAT = (
    TYPE = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    COMPRESSION = NONE
    NULL_IF = ('')
)
HEADER = TRUE
SINGLE = TRUE
MAX_FILE_SIZE = 5368709120
OVERWRITE = TRUE;


COPY INTO @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/west_midlands_police_clean.csv
FROM (
    SELECT *
    FROM CRIME_ETL_DB.DATA_QUALITY.STREET_CRIME_CLEAN
    WHERE FORCE_NAME = 'West Midlands Police'
)
FILE_FORMAT = (
    TYPE = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    COMPRESSION = NONE
    NULL_IF = ('')
)
HEADER = TRUE
SINGLE = TRUE
MAX_FILE_SIZE = 5368709120
OVERWRITE = TRUE;


COPY INTO @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/south_wales_police_clean.csv
FROM (
    SELECT *
    FROM CRIME_ETL_DB.DATA_QUALITY.STREET_CRIME_CLEAN
    WHERE FORCE_NAME = 'South Wales Police'
)
FILE_FORMAT = (
    TYPE = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    COMPRESSION = NONE
    NULL_IF = ('')
)
HEADER = TRUE
SINGLE = TRUE
MAX_FILE_SIZE = 5368709120
OVERWRITE = TRUE;


COPY INTO @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/sussex_police_clean.csv
FROM (
    SELECT *
    FROM CRIME_ETL_DB.DATA_QUALITY.STREET_CRIME_CLEAN
    WHERE FORCE_NAME = 'Sussex Police'
)
FILE_FORMAT = (
    TYPE = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    COMPRESSION = NONE
    NULL_IF = ('')
)
HEADER = TRUE
SINGLE = TRUE
MAX_FILE_SIZE = 5368709120
OVERWRITE = TRUE;

### Step 2: Supplementary Data Cleaning and Validation

#### Outline:

This step cleans the three enrichment sources used by the police-crime project:

1. ONS police-force-area population estimates, 2021-2024.
2. England Index of Multiple Deprivation (IMD) 2025.
3. Wales Index of Multiple Deprivation (WIMD) 2025.

It reads external Snowflake stages without modifying them and publishes separate clean, quarantine, audit, and validation tables. England IMD and Wales WIMD remain explicitly identified because their scores and ranks are not directly comparable.

The notebook does not perform the enrichment join. It prepares validated handoff files for the enrichment owner.

## Expected source grain

| Dataset | Raw structure | Clean grain |
|---|---|---|
| Population | One force/year row with 172 age-sex columns | `FORCE_NAME + YEAR` |
| England IMD | One row per 2021 LSOA | `LSOA_CODE` |
| Wales WIMD | Long format: LSOA × domain × measure | `LSOA_CODE` after pivot |

The England crime-domain measures and Wales community-safety measures are preserved
but clearly flagged as unsuitable for explaining police crime because they can create
circular analysis.

In [ ]:
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception as exc:
    raise RuntimeError(
        "Run this notebook inside Snowflake or provide a configured Snowpark Session."
    ) from exc

RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)

# Raw external stages
POPULATION_STAGE = "CRIME_ETL_DB.RAW.POPULATION_S3_STAGE"
ENGLAND_STAGE = "CRIME_ETL_DB.RAW.DEPRIVATION_ENGLAND_S3_STAGE"
WALES_STAGE = "CRIME_ETL_DB.RAW.DEPRIVATION_WALES_S3_STAGE"

# Published Snowflake tables
CLEAN_SCHEMA = "CRIME_ETL_DB.DATA_QUALITY"
POPULATION_CLEAN = f"{CLEAN_SCHEMA}.POPULATION_CLEAN"
POPULATION_QUARANTINE = f"{CLEAN_SCHEMA}.POPULATION_QUARANTINE"
ENGLAND_CLEAN = f"{CLEAN_SCHEMA}.DEPRIVATION_ENGLAND_CLEAN"
ENGLAND_QUARANTINE = f"{CLEAN_SCHEMA}.DEPRIVATION_ENGLAND_QUARANTINE"
WALES_LONG_CLEAN = f"{CLEAN_SCHEMA}.DEPRIVATION_WALES_LONG_CLEAN"
WALES_CLEAN = f"{CLEAN_SCHEMA}.DEPRIVATION_WALES_CLEAN"
WALES_QUARANTINE = f"{CLEAN_SCHEMA}.DEPRIVATION_WALES_QUARANTINE"
AUDIT_TABLE = f"{CLEAN_SCHEMA}.SUPPLEMENTARY_CLEANING_AUDIT"
VALIDATION_TABLE = f"{CLEAN_SCHEMA}.SUPPLEMENTARY_VALIDATION_RESULTS"

# Existing external clean stage shared with the police-data exports.
# Set EXPORT_TO_S3=False only when this stage is temporarily unavailable.
EXPORT_TO_S3 = True
CLEAN_EXPORT_STAGE = "CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE"

IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*(\.[A-Za-z_][A-Za-z0-9_]*){0,2}$")
for name in [
    POPULATION_STAGE, ENGLAND_STAGE, WALES_STAGE, CLEAN_SCHEMA,
    POPULATION_CLEAN, POPULATION_QUARANTINE, ENGLAND_CLEAN,
    ENGLAND_QUARANTINE, WALES_LONG_CLEAN, WALES_CLEAN,
    WALES_QUARANTINE, AUDIT_TABLE, VALIDATION_TABLE,
    CLEAN_EXPORT_STAGE,
]:
    if not IDENTIFIER.fullmatch(name):
        raise ValueError(f"Unsafe Snowflake identifier: {name}")

print(f"Supplementary cleaning run: {RUN_ID}")

## 1. Confirm source stages and create audit tables

In [ ]:

session.sql(f"CREATE SCHEMA IF NOT EXISTS {CLEAN_SCHEMA}").collect()

for stage in [POPULATION_STAGE, ENGLAND_STAGE, WALES_STAGE]:
    files = session.sql(f"LIST @{stage}").collect()
    if not files:
        raise ValueError(f"No files found in @{stage}")
    print(f"PASS - @{stage}: {len(files)} file(s)")

session.sql(f"""
CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
    RUN_ID STRING,
    DATASET STRING,
    STARTED_AT TIMESTAMP_TZ,
    COMPLETED_AT TIMESTAMP_TZ,
    RAW_ROWS NUMBER,
    CLEAN_ROWS NUMBER,
    QUARANTINE_ROWS NUMBER,
    STATUS STRING
)
""").collect()

session.sql(f"""
CREATE TABLE IF NOT EXISTS {VALIDATION_TABLE} (
    RUN_ID STRING,
    DATASET STRING,
    CHECK_NAME STRING,
    SEVERITY STRING,
    OBSERVED_VALUE STRING,
    EXPECTED_VALUE STRING,
    PASSED BOOLEAN,
    CHECKED_AT TIMESTAMP_TZ
)
""").collect()

## 2. Population cleaning

The ONS source contains 172 population columns: female ages `F0-F85` and male ages `M0-M85`. The final population is their sum. Only observed years 2021-2024 are published; 2025 and 2026 are not estimated in this cleaning layer.

In [ ]:
# Positional CSV columns: $1 code, $2 force, $3 year, $4-$175 age/sex values.
session.sql("USE DATABASE CRIME_ETL_DB").collect()
session.sql("USE SCHEMA RAW").collect()

age_positions = range(4, 176)
population_sum_sql = " + ".join(
    f"COALESCE(TRY_TO_NUMBER(t.${position}), 0)" for position in age_positions
)
invalid_age_sql = " + ".join(
    "IFF(NULLIF(TRIM(t.${0}), '') IS NOT NULL "
    "AND TRY_TO_NUMBER(t.${0}) IS NULL, 1, 0)".format(position)
    for position in age_positions
)

session.sql(f"""
CREATE OR REPLACE TEMPORARY TABLE STG_POPULATION AS
WITH PARSED AS (
    SELECT
        UPPER(NULLIF(TRIM(t.$1), '')) AS PFA_CODE,
        NULLIF(TRIM(t.$2), '') AS FORCE_NAME_RAW,
        TRY_TO_NUMBER(t.$3) AS YEAR,
        ({population_sum_sql}) AS TOTAL_POPULATION,
        ({invalid_age_sql}) AS INVALID_AGE_CELLS,
        METADATA$FILENAME AS SOURCE_FILE_NAME
    FROM @{POPULATION_STAGE} t
    WHERE TRY_TO_NUMBER(t.$3) BETWEEN 2021 AND 2024
), STANDARDISED AS (
    SELECT *,
        CASE LOWER(FORCE_NAME_RAW)
            WHEN 'metropolitan police' THEN 'Metropolitan Police Service'
            WHEN 'metropolitan police service' THEN 'Metropolitan Police Service'
            WHEN 'west midlands' THEN 'West Midlands Police'
            WHEN 'west midlands police' THEN 'West Midlands Police'
            WHEN 'south wales' THEN 'South Wales Police'
            WHEN 'south wales police' THEN 'South Wales Police'
            WHEN 'sussex' THEN 'Sussex Police'
            WHEN 'sussex police' THEN 'Sussex Police'
        END AS FORCE_NAME
    FROM PARSED
), ASSESSED AS (
    SELECT *,
        ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
            IFF(PFA_CODE IS NULL, 'MISSING_PFA_CODE', NULL),
            IFF(FORCE_NAME IS NULL, 'OUT_OF_SCOPE_OR_UNKNOWN_FORCE', NULL),
            IFF(YEAR NOT BETWEEN 2021 AND 2024, 'INVALID_YEAR', NULL),
            IFF(INVALID_AGE_CELLS > 0, 'NON_NUMERIC_AGE_VALUE', NULL),
            IFF(TOTAL_POPULATION <= 0, 'INVALID_TOTAL_POPULATION', NULL)
        ), '|') AS BASE_REJECTION_REASON,
        ROW_NUMBER() OVER (
            PARTITION BY FORCE_NAME, YEAR
            ORDER BY SOURCE_FILE_NAME
        ) AS GRAIN_RANK
    FROM STANDARDISED
    WHERE FORCE_NAME IS NOT NULL
)
SELECT *,
    CASE
        WHEN BASE_REJECTION_REASON <> '' AND GRAIN_RANK > 1
            THEN BASE_REJECTION_REASON || '|DUPLICATE_FORCE_YEAR'
        WHEN BASE_REJECTION_REASON <> '' THEN BASE_REJECTION_REASON
        WHEN GRAIN_RANK > 1 THEN 'DUPLICATE_FORCE_YEAR'
    END AS REJECTION_REASON
FROM ASSESSED
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {POPULATION_CLEAN} AS
SELECT
    PFA_CODE, FORCE_NAME, YEAR::INTEGER AS YEAR,
    TOTAL_POPULATION::INTEGER AS TOTAL_POPULATION,
    FALSE AS IS_ESTIMATED,
    SOURCE_FILE_NAME,
    '{RUN_ID}' AS CLEANING_RUN_ID,
    CURRENT_TIMESTAMP() AS CLEANED_AT
FROM STG_POPULATION
WHERE REJECTION_REASON IS NULL
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {POPULATION_QUARANTINE} AS
SELECT *, '{RUN_ID}' AS CLEANING_RUN_ID, CURRENT_TIMESTAMP() AS QUARANTINED_AT
FROM STG_POPULATION
WHERE REJECTION_REASON IS NOT NULL
""").collect()

## 3. England IMD cleaning

The clean handoff retains the overall index and non-crime socioeconomic domains.
Crime-domain fields are preserved with an exclusion flag but should not be used as
independent explanatory variables for police crime.

In [ ]:
session.sql(f"""
CREATE OR REPLACE TEMPORARY TABLE STG_ENGLAND_IMD AS
WITH PARSED AS (
    SELECT
        UPPER(NULLIF(TRIM(t.$1), '')) AS LSOA_CODE,
        NULLIF(TRIM(t.$2), '') AS LSOA_NAME,
        UPPER(NULLIF(TRIM(t.$3), '')) AS LOCAL_AUTHORITY_CODE,
        NULLIF(TRIM(t.$4), '') AS LOCAL_AUTHORITY_NAME,
        TRY_TO_DECIMAL(t.$5, 10, 3) AS IMD_SCORE,
        TRY_TO_NUMBER(t.$6) AS IMD_RANK,
        TRY_TO_NUMBER(t.$7) AS IMD_DECILE,
        TRY_TO_DECIMAL(t.$8, 10, 3) AS INCOME_SCORE,
        TRY_TO_NUMBER(t.$9) AS INCOME_RANK,
        TRY_TO_NUMBER(t.$10) AS INCOME_DECILE,
        TRY_TO_DECIMAL(t.$11, 10, 3) AS EMPLOYMENT_SCORE,
        TRY_TO_NUMBER(t.$12) AS EMPLOYMENT_RANK,
        TRY_TO_NUMBER(t.$13) AS EMPLOYMENT_DECILE,
        TRY_TO_DECIMAL(t.$14, 10, 3) AS EDUCATION_SCORE,
        TRY_TO_NUMBER(t.$15) AS EDUCATION_RANK,
        TRY_TO_NUMBER(t.$16) AS EDUCATION_DECILE,
        TRY_TO_DECIMAL(t.$17, 10, 3) AS HEALTH_SCORE,
        TRY_TO_NUMBER(t.$18) AS HEALTH_RANK,
        TRY_TO_NUMBER(t.$19) AS HEALTH_DECILE,
        TRY_TO_DECIMAL(t.$20, 10, 3) AS CRIME_SCORE,
        TRY_TO_NUMBER(t.$21) AS CRIME_RANK,
        TRY_TO_NUMBER(t.$22) AS CRIME_DECILE,
        TRY_TO_DECIMAL(t.$23, 10, 3) AS HOUSING_BARRIERS_SCORE,
        TRY_TO_NUMBER(t.$24) AS HOUSING_BARRIERS_RANK,
        TRY_TO_NUMBER(t.$25) AS HOUSING_BARRIERS_DECILE,
        TRY_TO_DECIMAL(t.$26, 10, 3) AS LIVING_ENVIRONMENT_SCORE,
        TRY_TO_NUMBER(t.$27) AS LIVING_ENVIRONMENT_RANK,
        TRY_TO_NUMBER(t.$28) AS LIVING_ENVIRONMENT_DECILE,
        TRY_TO_NUMBER(t.$53) AS TOTAL_POPULATION_MID_2022,
        TRY_TO_NUMBER(t.$54) AS DEPENDENT_CHILDREN_0_15_MID_2022,
        TRY_TO_NUMBER(t.$55) AS OLDER_POPULATION_60_PLUS_MID_2022,
        TRY_TO_NUMBER(t.$56) AS WORKING_AGE_POPULATION_MID_2022,
        METADATA$FILENAME AS SOURCE_FILE_NAME
    FROM @{ENGLAND_STAGE} t
    WHERE NULLIF(TRIM(t.$1), '') IS NOT NULL
      AND LOWER(TRIM(t.$1)) <> 'lsoa code (2021)'
), ASSESSED AS (
    SELECT *,
        ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
            IFF(NOT REGEXP_LIKE(LSOA_CODE, '^E01[0-9]{{6}}$'), 'INVALID_LSOA_CODE', NULL),
            IFF(LSOA_NAME IS NULL, 'MISSING_LSOA_NAME', NULL),
            IFF(IMD_SCORE IS NULL, 'INVALID_IMD_SCORE', NULL),
            IFF(IMD_RANK IS NULL OR IMD_RANK < 1, 'INVALID_IMD_RANK', NULL),
            IFF(IMD_DECILE NOT BETWEEN 1 AND 10, 'INVALID_IMD_DECILE', NULL),
            IFF(INCOME_DECILE NOT BETWEEN 1 AND 10, 'INVALID_INCOME_DECILE', NULL),
            IFF(EMPLOYMENT_DECILE NOT BETWEEN 1 AND 10, 'INVALID_EMPLOYMENT_DECILE', NULL),
            IFF(EDUCATION_DECILE NOT BETWEEN 1 AND 10, 'INVALID_EDUCATION_DECILE', NULL),
            IFF(HEALTH_DECILE NOT BETWEEN 1 AND 10, 'INVALID_HEALTH_DECILE', NULL),
            IFF(HOUSING_BARRIERS_DECILE NOT BETWEEN 1 AND 10,
                'INVALID_HOUSING_BARRIERS_DECILE', NULL),
            IFF(LIVING_ENVIRONMENT_DECILE NOT BETWEEN 1 AND 10,
                'INVALID_LIVING_ENVIRONMENT_DECILE', NULL),
            IFF(TOTAL_POPULATION_MID_2022 IS NULL OR TOTAL_POPULATION_MID_2022 < 0,
                'INVALID_POPULATION', NULL)
        ), '|') AS BASE_REJECTION_REASON,
        ROW_NUMBER() OVER (PARTITION BY LSOA_CODE ORDER BY SOURCE_FILE_NAME) AS LSOA_RANK
    FROM PARSED
)
SELECT *,
    CASE
        WHEN BASE_REJECTION_REASON <> '' AND LSOA_RANK > 1
            THEN BASE_REJECTION_REASON || '|DUPLICATE_LSOA'
        WHEN BASE_REJECTION_REASON <> '' THEN BASE_REJECTION_REASON
        WHEN LSOA_RANK > 1 THEN 'DUPLICATE_LSOA'
    END AS REJECTION_REASON
FROM ASSESSED
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {ENGLAND_CLEAN} AS
SELECT
    LSOA_CODE, LSOA_NAME, LOCAL_AUTHORITY_CODE, LOCAL_AUTHORITY_NAME,
    IMD_SCORE, IMD_RANK::INTEGER AS IMD_RANK, IMD_DECILE::INTEGER AS IMD_DECILE,
    INCOME_SCORE, INCOME_RANK::INTEGER AS INCOME_RANK,
    INCOME_DECILE::INTEGER AS INCOME_DECILE,
    EMPLOYMENT_SCORE, EMPLOYMENT_RANK::INTEGER AS EMPLOYMENT_RANK,
    EMPLOYMENT_DECILE::INTEGER AS EMPLOYMENT_DECILE,
    EDUCATION_SCORE, EDUCATION_RANK::INTEGER AS EDUCATION_RANK,
    EDUCATION_DECILE::INTEGER AS EDUCATION_DECILE,
    HEALTH_SCORE, HEALTH_RANK::INTEGER AS HEALTH_RANK,
    HEALTH_DECILE::INTEGER AS HEALTH_DECILE,
    CRIME_SCORE, CRIME_RANK::INTEGER AS CRIME_RANK,
    CRIME_DECILE::INTEGER AS CRIME_DECILE,
    TRUE AS EXCLUDE_CRIME_DOMAIN_FROM_CRIME_ANALYSIS,
    HOUSING_BARRIERS_SCORE,
    HOUSING_BARRIERS_RANK::INTEGER AS HOUSING_BARRIERS_RANK,
    HOUSING_BARRIERS_DECILE::INTEGER AS HOUSING_BARRIERS_DECILE,
    LIVING_ENVIRONMENT_SCORE,
    LIVING_ENVIRONMENT_RANK::INTEGER AS LIVING_ENVIRONMENT_RANK,
    LIVING_ENVIRONMENT_DECILE::INTEGER AS LIVING_ENVIRONMENT_DECILE,
    TOTAL_POPULATION_MID_2022::INTEGER AS TOTAL_POPULATION_MID_2022,
    DEPENDENT_CHILDREN_0_15_MID_2022::INTEGER AS DEPENDENT_CHILDREN_0_15_MID_2022,
    OLDER_POPULATION_60_PLUS_MID_2022::INTEGER AS OLDER_POPULATION_60_PLUS_MID_2022,
    WORKING_AGE_POPULATION_MID_2022::INTEGER AS WORKING_AGE_POPULATION_MID_2022,
    'IMD' AS INDEX_NAME, 2025 AS INDEX_YEAR, 'England' AS NATION,
    'LSOA 2021' AS GEOGRAPHY_VERSION,
    SOURCE_FILE_NAME, '{RUN_ID}' AS CLEANING_RUN_ID,
    CURRENT_TIMESTAMP() AS CLEANED_AT
FROM STG_ENGLAND_IMD
WHERE REJECTION_REASON IS NULL
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {ENGLAND_QUARANTINE} AS
SELECT *, '{RUN_ID}' AS CLEANING_RUN_ID, CURRENT_TIMESTAMP() AS QUARANTINED_AT
FROM STG_ENGLAND_IMD
WHERE REJECTION_REASON IS NOT NULL
""").collect()

## 4. Wales WIMD cleaning and pivot

The Wales source contains 86,265 expected rows: 1,917 LSOAs × 9 domains × 5
measures. The long clean table preserves all source measures. The pivoted table
contains ranks and deciles for enrichment.

In [ ]:
session.sql(f"""
CREATE OR REPLACE TEMPORARY TABLE STG_WALES_WIMD AS
WITH PARSED AS (
    SELECT
        TRY_TO_NUMBER(t.$1) AS MEASURE_VALUE,
        INITCAP(NULLIF(TRIM(t.$3), '')) AS MEASURE,
        UPPER(NULLIF(TRIM(t.$7), '')) AS LSOA_CODE,
        NULLIF(TRIM(t.$11), '') AS LSOA_NAME,
        NULLIF(TRIM(t.$15), '') AS DOMAIN,
        NULLIF(TRIM(t.$19), '') AS NOTES,
        METADATA$FILENAME AS SOURCE_FILE_NAME
    FROM @{WALES_STAGE} t
    WHERE NULLIF(TRIM(t.$7), '') IS NOT NULL
      AND LOWER(TRIM(t.$7)) <> 'area code'
), ASSESSED AS (
    SELECT *,
        ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
            IFF(NOT REGEXP_LIKE(LSOA_CODE, '^W01[0-9]{{6}}$'), 'INVALID_LSOA_CODE', NULL),
            IFF(LSOA_NAME IS NULL, 'MISSING_LSOA_NAME', NULL),
            IFF(DOMAIN IS NULL, 'MISSING_DOMAIN', NULL),
            IFF(MEASURE NOT IN ('Rank','Decile','Group','Quartile','Quintile'),
                'UNKNOWN_MEASURE', NULL),
            IFF(MEASURE_VALUE IS NULL, 'NON_NUMERIC_VALUE', NULL),
            IFF(MEASURE = 'Rank' AND MEASURE_VALUE NOT BETWEEN 1 AND 1917,
                'RANK_OUT_OF_RANGE', NULL),
            IFF(MEASURE = 'Decile' AND MEASURE_VALUE NOT BETWEEN 1 AND 10,
                'DECILE_OUT_OF_RANGE', NULL),
            IFF(MEASURE = 'Group' AND MEASURE_VALUE NOT BETWEEN 1 AND 5,
                'GROUP_OUT_OF_RANGE', NULL),
            IFF(MEASURE = 'Quartile' AND MEASURE_VALUE NOT BETWEEN 1 AND 4,
                'QUARTILE_OUT_OF_RANGE', NULL),
            IFF(MEASURE = 'Quintile' AND MEASURE_VALUE NOT BETWEEN 1 AND 5,
                'QUINTILE_OUT_OF_RANGE', NULL)
        ), '|') AS BASE_REJECTION_REASON,
        ROW_NUMBER() OVER (
            PARTITION BY LSOA_CODE, DOMAIN, MEASURE
            ORDER BY SOURCE_FILE_NAME
        ) AS MEASURE_RANK
    FROM PARSED
)
SELECT *,
    CASE
        WHEN BASE_REJECTION_REASON <> '' AND MEASURE_RANK > 1
            THEN BASE_REJECTION_REASON || '|DUPLICATE_LSOA_DOMAIN_MEASURE'
        WHEN BASE_REJECTION_REASON <> '' THEN BASE_REJECTION_REASON
        WHEN MEASURE_RANK > 1 THEN 'DUPLICATE_LSOA_DOMAIN_MEASURE'
    END AS REJECTION_REASON
FROM ASSESSED
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {WALES_LONG_CLEAN} AS
SELECT
    LSOA_CODE, LSOA_NAME, DOMAIN, MEASURE,
    MEASURE_VALUE::INTEGER AS MEASURE_VALUE,
    NOTES, 'WIMD' AS INDEX_NAME, 2025 AS INDEX_YEAR, 'Wales' AS NATION,
    SOURCE_FILE_NAME, '{RUN_ID}' AS CLEANING_RUN_ID,
    CURRENT_TIMESTAMP() AS CLEANED_AT
FROM STG_WALES_WIMD
WHERE REJECTION_REASON IS NULL
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {WALES_QUARANTINE} AS
SELECT *, '{RUN_ID}' AS CLEANING_RUN_ID, CURRENT_TIMESTAMP() AS QUARANTINED_AT
FROM STG_WALES_WIMD
WHERE REJECTION_REASON IS NOT NULL
""").collect()

domains = {
    "WIMD": "WIMD",
    "Income": "INCOME",
    "Employment": "EMPLOYMENT",
    "Education": "EDUCATION",
    "Health": "HEALTH",
    "Access to services": "ACCESS_TO_SERVICES",
    "Housing": "HOUSING",
    "Physical environment": "PHYSICAL_ENVIRONMENT",
    "Community safety": "COMMUNITY_SAFETY",
}
pivot_columns = []
for source_domain, output_prefix in domains.items():
    safe_domain = source_domain.replace("'", "''")
    for measure in ["Rank", "Decile"]:
        pivot_columns.append(
            f"MAX(IFF(DOMAIN = '{safe_domain}' AND MEASURE = '{measure}', "
            f"MEASURE_VALUE, NULL))::{ 'INTEGER' } AS {output_prefix}_{measure.upper()}"
        )

session.sql(f"""
CREATE OR REPLACE TABLE {WALES_CLEAN} AS
SELECT
    LSOA_CODE,
    MAX(LSOA_NAME) AS LSOA_NAME,
    {", ".join(pivot_columns)},
    TRUE AS EXCLUDE_COMMUNITY_SAFETY_FROM_CRIME_ANALYSIS,
    'WIMD' AS INDEX_NAME,
    2025 AS INDEX_YEAR,
    'Wales' AS NATION,
    'Confirm source LSOA boundary version before joining' AS GEOGRAPHY_VERSION,
    '{RUN_ID}' AS CLEANING_RUN_ID,
    CURRENT_TIMESTAMP() AS CLEANED_AT
FROM {WALES_LONG_CLEAN}
GROUP BY LSOA_CODE
""").collect()

## 5. Validation and audit

In [ ]:
checks = []

def add_check(dataset, name, severity, observed, expected, passed):
    record = {
        "dataset": dataset, "name": name, "severity": severity,
        "observed": str(observed), "expected": str(expected), "passed": bool(passed),
    }
    checks.append(record)
    observed_sql = record["observed"].replace("'", "''")
    expected_sql = record["expected"].replace("'", "''")
    session.sql(f"""
    INSERT INTO {VALIDATION_TABLE}
    SELECT '{RUN_ID}', '{dataset}', '{name}', '{severity}',
           '{observed_sql}', '{expected_sql}', {str(record["passed"]).upper()},
           CURRENT_TIMESTAMP()
    """).collect()

def scalar(sql, column="N"):
    return session.sql(sql).collect()[0][column]

# Population validations
pop_clean_rows = session.table(POPULATION_CLEAN).count()
pop_quarantine_rows = session.table(POPULATION_QUARANTINE).count()
pop_raw_rows = session.table("STG_POPULATION").count()
add_check("population", "row_reconciliation", "CRITICAL",
          pop_clean_rows + pop_quarantine_rows, pop_raw_rows,
          pop_clean_rows + pop_quarantine_rows == pop_raw_rows)
add_check("population", "expected_16_rows", "CRITICAL",
          pop_clean_rows, 16, pop_clean_rows == 16)
pop_duplicates = scalar(f"""
SELECT COUNT(*) AS N FROM (
    SELECT FORCE_NAME, YEAR FROM {POPULATION_CLEAN}
    GROUP BY FORCE_NAME, YEAR HAVING COUNT(*) > 1
)""")
add_check("population", "unique_force_year", "CRITICAL",
          pop_duplicates, 0, pop_duplicates == 0)

# England validations
eng_clean_rows = session.table(ENGLAND_CLEAN).count()
eng_quarantine_rows = session.table(ENGLAND_QUARANTINE).count()
eng_raw_rows = session.table("STG_ENGLAND_IMD").count()
add_check("england_imd", "row_reconciliation", "CRITICAL",
          eng_clean_rows + eng_quarantine_rows, eng_raw_rows,
          eng_clean_rows + eng_quarantine_rows == eng_raw_rows)
eng_duplicates = scalar(f"""
SELECT COUNT(*) AS N FROM (
    SELECT LSOA_CODE FROM {ENGLAND_CLEAN}
    GROUP BY LSOA_CODE HAVING COUNT(*) > 1
)""")
add_check("england_imd", "unique_lsoa", "CRITICAL",
          eng_duplicates, 0, eng_duplicates == 0)
eng_invalid_deciles = scalar(f"""
SELECT COUNT(*) AS N FROM {ENGLAND_CLEAN}
WHERE IMD_DECILE NOT BETWEEN 1 AND 10
   OR INCOME_DECILE NOT BETWEEN 1 AND 10
   OR EMPLOYMENT_DECILE NOT BETWEEN 1 AND 10
   OR EDUCATION_DECILE NOT BETWEEN 1 AND 10
   OR HEALTH_DECILE NOT BETWEEN 1 AND 10
   OR HOUSING_BARRIERS_DECILE NOT BETWEEN 1 AND 10
   OR LIVING_ENVIRONMENT_DECILE NOT BETWEEN 1 AND 10
""")
add_check("england_imd", "decile_ranges", "CRITICAL",
          eng_invalid_deciles, 0, eng_invalid_deciles == 0)

# Wales validations
wales_long_rows = session.table(WALES_LONG_CLEAN).count()
wales_quarantine_rows = session.table(WALES_QUARANTINE).count()
wales_raw_rows = session.table("STG_WALES_WIMD").count()
add_check("wales_wimd", "row_reconciliation", "CRITICAL",
          wales_long_rows + wales_quarantine_rows, wales_raw_rows,
          wales_long_rows + wales_quarantine_rows == wales_raw_rows)
add_check("wales_wimd", "expected_86265_long_rows", "CRITICAL",
          wales_long_rows, 86265, wales_long_rows == 86265)
wales_clean_rows = session.table(WALES_CLEAN).count()
add_check("wales_wimd", "expected_1917_lsoas", "CRITICAL",
          wales_clean_rows, 1917, wales_clean_rows == 1917)
wales_incomplete = scalar(f"""
SELECT COUNT(*) AS N FROM {WALES_CLEAN}
WHERE WIMD_RANK IS NULL OR WIMD_DECILE IS NULL
   OR INCOME_RANK IS NULL OR INCOME_DECILE IS NULL
   OR EMPLOYMENT_RANK IS NULL OR EMPLOYMENT_DECILE IS NULL
   OR EDUCATION_RANK IS NULL OR EDUCATION_DECILE IS NULL
   OR HEALTH_RANK IS NULL OR HEALTH_DECILE IS NULL
""")
add_check("wales_wimd", "core_pivot_completeness", "CRITICAL",
          wales_incomplete, 0, wales_incomplete == 0)

for dataset, raw_rows, clean_rows, quarantine_rows in [
    ("population", pop_raw_rows, pop_clean_rows, pop_quarantine_rows),
    ("england_imd", eng_raw_rows, eng_clean_rows, eng_quarantine_rows),
    ("wales_wimd", wales_raw_rows, wales_clean_rows, wales_quarantine_rows),
]:
    failed = [c for c in checks if c["dataset"] == dataset
              and c["severity"] == "CRITICAL" and not c["passed"]]
    status = "VALIDATED" if not failed else "VALIDATION_FAILED"
    session.sql(f"""
    INSERT INTO {AUDIT_TABLE}
    SELECT '{RUN_ID}', '{dataset}', TO_TIMESTAMP_TZ('{RUN_STARTED_AT.isoformat()}'),
           CURRENT_TIMESTAMP(), {raw_rows}, {clean_rows}, {quarantine_rows}, '{status}'
    """).collect()

session.sql(f"""
SELECT DATASET, CHECK_NAME, SEVERITY, OBSERVED_VALUE, EXPECTED_VALUE, PASSED
FROM {VALIDATION_TABLE}
WHERE RUN_ID = '{RUN_ID}'
ORDER BY DATASET, CHECK_NAME
""").show()

critical_failures = [c for c in checks if c["severity"] == "CRITICAL" and not c["passed"]]
if critical_failures:
    raise AssertionError(
        "Critical validation failure(s): "
        + ", ".join(f'{c["dataset"]}.{c["name"]}' for c in critical_failures)
    )
print("PASS - all supplementary datasets validated")

## 6. Export validated handoff files

Exports occur only after all critical checks pass. All supplementary files are written
to the existing clean stage used by the police-data pipeline.

In [ ]:
if EXPORT_TO_S3:
    exports = [
        (POPULATION_CLEAN, "population_clean.csv"),
        (ENGLAND_CLEAN, "deprivation_england_clean.csv"),
        (WALES_CLEAN, "deprivation_wales_clean.csv"),
    ]
    for table_name, file_name in exports:
        session.sql(f"""
        COPY INTO @{CLEAN_EXPORT_STAGE}/{file_name}
        FROM {table_name}
        FILE_FORMAT = (
            TYPE = CSV
            FIELD_OPTIONALLY_ENCLOSED_BY = '"'
            COMPRESSION = NONE
            NULL_IF = ('')
        )
        HEADER = TRUE
        SINGLE = TRUE
        OVERWRITE = TRUE
        """).collect()
        print(f"Exported @{CLEAN_EXPORT_STAGE}/{file_name}")

    session.sql(f"""
    COPY INTO @{CLEAN_EXPORT_STAGE}/supplementary_cleaning_audit.csv
    FROM (
        SELECT *
        FROM {AUDIT_TABLE}
        WHERE RUN_ID = '{RUN_ID}'
        ORDER BY DATASET
    )
    FILE_FORMAT = (
        TYPE = CSV
        FIELD_OPTIONALLY_ENCLOSED_BY = '"'
        COMPRESSION = NONE
        NULL_IF = ('')
    )
    HEADER = TRUE
    SINGLE = TRUE
    OVERWRITE = TRUE
    """).collect()

    session.sql(f"LIST @{CLEAN_EXPORT_STAGE}").show()
else:
    print("S3 export disabled. Validated Snowflake tables were still published.")

## Handoff notes

- Population is observed only for 2021-2024; `IS_ESTIMATED` is always false.
- England IMD uses 2021 LSOA geography.
- Confirm the Wales LSOA boundary version before joining.
- Never compare raw England IMD scores directly with Wales WIMD ranks.
- Exclude England crime-domain and Wales community-safety measures when analysing
  police crime to avoid circularity.
- Population must join on `FORCE_NAME + YEAR`.
- Deprivation must join on `LSOA_CODE`.
- Before and after each left join, reconcile crime row counts and report match rates.

## Feature Engineering & Transformation Layer

### Step 1: Create Tables for Cleaned Data

#### Outline: 

Tables for the crime and enrichment data are created and loaded with the clean data.

In [ ]:
USE DATABASE CRIME_ETL_DB;
USE SCHEMA CLEAN;

CREATE FILE FORMAT IF NOT EXISTS CLEAN_CSV_FORMAT
    TYPE = CSV
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    EMPTY_FIELD_AS_NULL = TRUE
    NULL_IF = ('', 'NULL', 'null');

In [ ]:
USE DATABASE CRIME_ETL_DB;
USE SCHEMA CLEAN;

-- Creating the table for the cleaned crime data
CREATE OR REPLACE TABLE CRIME_DATA_CLEAN (
    CRIME_ID VARCHAR,
    YEAR VARCHAR,
    MONTH VARCHAR,
    REPORTED_BY VARCHAR,
    FALLS_WITHIN VARCHAR,
    LONGITUDE FLOAT,
    LATITUDE FLOAT,
    LOCATION VARCHAR,
    LSOA_CODE VARCHAR,
    LSOA_NAME VARCHAR,
    CRIME_TYPE VARCHAR
);

In [ ]:
-- Copying the cleaned Sussex crime data from the Amazon s3 bucket to the table
COPY INTO CRIME_DATA_CLEAN (
    CRIME_ID,
    YEAR,
    MONTH,
    REPORTED_BY,
    FALLS_WITHIN,
    LONGITUDE,
    LATITUDE,
    LOCATION,
    LSOA_CODE,
    LSOA_NAME,
    CRIME_TYPE
)
FROM (
    SELECT 
        $1,                     
        $3,                     
        $4,
        $5,                      
        $6,                        
        TRY_CAST($7 AS FLOAT),     
        TRY_CAST($8 AS FLOAT),    
        $9,                        
        $10,                       
        $11,                      
        $12                       
    FROM @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/sussex_police_clean.csv
)
FILE_FORMAT = (FORMAT_NAME = 'CLEAN_CSV_FORMAT')
ON_ERROR = 'ABORT_STATEMENT'
FORCE = TRUE;

In [ ]:
-- Copying the cleaned South Wales crime data from the Amazon s3 bucket to the table
COPY INTO CRIME_DATA_CLEAN (
    CRIME_ID,
    YEAR,
    MONTH,
    REPORTED_BY,
    FALLS_WITHIN,
    LONGITUDE,
    LATITUDE,
    LOCATION,
    LSOA_CODE,
    LSOA_NAME,
    CRIME_TYPE
)
FROM (
    SELECT 
        $1,                        -- CRIME_ID
        $3,                        -- MONTH
        $4,
        $5,                        -- REPORTED_BY ("Metropolitan Police Service")
        $6,                        -- FALLS_WITHIN ("Metropolitan Police Service")
        TRY_CAST($7 AS FLOAT),     -- LONGITUDE (0.177936)
        TRY_CAST($8 AS FLOAT),     -- LATITUDE (51.45086)
        $9,                        -- LOCATION
        $10,                       -- LSOA_CODE
        $11,                       -- LSOA_NAME
        $12                       -- CRIME_TYPE
    FROM @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/south_wales_police_clean.csv
)
FILE_FORMAT = (FORMAT_NAME = 'CLEAN_CSV_FORMAT')
ON_ERROR = 'ABORT_STATEMENT'
FORCE = TRUE;

In [ ]:
-- Copying the cleaned West Midlands crime data from the Amazon s3 bucket to the table
COPY INTO CRIME_DATA_CLEAN (
    CRIME_ID,
    YEAR,
    MONTH,
    REPORTED_BY,
    FALLS_WITHIN,
    LONGITUDE,
    LATITUDE,
    LOCATION,
    LSOA_CODE,
    LSOA_NAME,
    CRIME_TYPE
)
FROM (
    SELECT 
        $1,                        -- CRIME_ID
        $3,                        -- MONTH
        $4,
        $5,                        -- REPORTED_BY ("Metropolitan Police Service")
        $6,                        -- FALLS_WITHIN ("Metropolitan Police Service")
        TRY_CAST($7 AS FLOAT),     -- LONGITUDE (0.177936)
        TRY_CAST($8 AS FLOAT),     -- LATITUDE (51.45086)
        $9,                        -- LOCATION
        $10,                       -- LSOA_CODE
        $11,                       -- LSOA_NAME
        $12                       -- CRIME_TYPE
    FROM @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/west_midlands_police_clean.csv
)
FILE_FORMAT = (FORMAT_NAME = 'CLEAN_CSV_FORMAT')
ON_ERROR = 'ABORT_STATEMENT'
FORCE = TRUE;


In [ ]:
-- Copying the cleaned Metropolitan crime data from the Amazon s3 bucket to the table
COPY INTO CRIME_DATA_CLEAN (
    CRIME_ID,
    YEAR,
    MONTH,
    REPORTED_BY,
    FALLS_WITHIN,
    LONGITUDE,
    LATITUDE,
    LOCATION,
    LSOA_CODE,
    LSOA_NAME,
    CRIME_TYPE
)
FROM (
    SELECT 
        $1,                        -- CRIME_ID
        $3,                        -- MONTH
        $4,
        $5,                        -- REPORTED_BY ("Metropolitan Police Service")
        $6,                        -- FALLS_WITHIN ("Metropolitan Police Service")
        TRY_CAST($7 AS FLOAT),     -- LONGITUDE (0.177936)
        TRY_CAST($8 AS FLOAT),     -- LATITUDE (51.45086)
        $9,                        -- LOCATION
        $10,                       -- LSOA_CODE
        $11,                       -- LSOA_NAME
        $12                       -- CRIME_TYPE
    FROM @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/metropolitan_police_clean.csv
)
FILE_FORMAT = (FORMAT_NAME = 'CLEAN_CSV_FORMAT')
ON_ERROR = 'ABORT_STATEMENT'
FORCE = TRUE;


In [ ]:
-- Creating the table for the cleaned population
create or replace TABLE POPULATION_CLEAN (
	PFA_CODE VARCHAR,
	FORCE_NAME VARCHAR,
	YEAR NUMBER,
	TOTAL_POPULATION NUMBER
);

-- Copying the cleaned population data from the Amazon s3 bucket to the table
COPY INTO POPULATION_CLEAN (
	PFA_CODE,
	FORCE_NAME,
	YEAR,
	TOTAL_POPULATION
)
FROM (
    SELECT 
        $1,                        
        $2,                        
        $3,
        $4
    FROM @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/population_clean.csv
)
FILE_FORMAT = (FORMAT_NAME = 'CLEAN_CSV_FORMAT')
ON_ERROR = 'ABORT_STATEMENT'
FORCE = TRUE;

In [ ]:
-- Creating the table for the cleaned Welsh deprivation data
CREATE OR REPLACE TABLE DEPRIVATION_WALES_CLEAN (
	LSOA_CODE VARCHAR,
	LSOA_NAME VARCHAR,
	WIMD_RANK NUMBER,
	WIMD_DECILE NUMBER,
	INCOME_RANK NUMBER,
	INCOME_DECILE NUMBER,
	EMPLOYMENT_RANK NUMBER,
	EMPLOYMENT_DECILE NUMBER,
	EDUCATION_RANK NUMBER,
	EDUCATION_DECILE NUMBER,
	HEALTH_RANK NUMBER,
	HEALTH_DECILE NUMBER,
	ACCESS_TO_SERVICES_RANK NUMBER,
	ACCESS_TO_SERVICES_DECILE NUMBER,
	HOUSING_RANK NUMBER,
	HOUSING_DECILE NUMBER,
	PHYSICAL_ENVIRONMENT_RANK NUMBER,
	PHYSICAL_ENVIRONMENT_DECILE NUMBER,
	COMMUNITY_SAFETY_RANK NUMBER,
	COMMUNITY_SAFETY_DECILE NUMBER
);

-- Copying the cleaned Welsh deprivation data data from the Amazon s3 bucket to the table
COPY INTO DEPRIVATION_WALES_CLEAN (
	LSOA_CODE,
	LSOA_NAME,
	WIMD_RANK,
	WIMD_DECILE,
	INCOME_RANK,
	INCOME_DECILE,
	EMPLOYMENT_RANK,
	EMPLOYMENT_DECILE,
	EDUCATION_RANK,
	EDUCATION_DECILE,
	HEALTH_RANK,
	HEALTH_DECILE,
	ACCESS_TO_SERVICES_RANK,
	ACCESS_TO_SERVICES_DECILE,
	HOUSING_RANK,
	HOUSING_DECILE,
	PHYSICAL_ENVIRONMENT_RANK,
	PHYSICAL_ENVIRONMENT_DECILE,
	COMMUNITY_SAFETY_RANK,
	COMMUNITY_SAFETY_DECILE 
)
FROM (
    SELECT 
        $1,                        
        $2,                        
        $3,
        $4,
        $5,                        
        $6,                        
        $7,
        $8,
        $9,                        
        $10,                        
        $11,
        $12,
        $13,                        
        $14,                        
        $15,
        $16,
        $17,                        
        $18,                        
        $19,
        $20
    FROM @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/deprivation_wales_clean.csv
)
FILE_FORMAT = (FORMAT_NAME = 'CLEAN_CSV_FORMAT')
ON_ERROR = 'ABORT_STATEMENT'
FORCE = TRUE;


In [ ]:
-- Creating the table for the cleaned English deprivation data
create or replace TABLE CRIME_ETL_DB.CLEAN.DEPRIVATION_ENGLAND_CLEAN (
	LSOA_CODE VARCHAR,
	LSOA_NAME VARCHAR,
	LOCAL_AUTHORITY_CODE VARCHAR,
	LOCAL_AUTHORITY_NAME VARCHAR,
	IMD_SCORE NUMBER,
	IMD_RANK NUMBER,
	IMD_DECILE NUMBER,
	INCOME_SCORE NUMBER,
	INCOME_RANK NUMBER,
	INCOME_DECILE NUMBER,
	EMPLOYMENT_SCORE NUMBER,
	EMPLOYMENT_RANK NUMBER,
	EMPLOYMENT_DECILE NUMBER,
	EDUCATION_SCORE NUMBER,
	EDUCATION_RANK NUMBER,
	EDUCATION_DECILE NUMBER,
	HEALTH_SCORE NUMBER,
	HEALTH_RANK NUMBER,
	HEALTH_DECILE NUMBER,
	CRIME_SCORE NUMBER,
	CRIME_RANK NUMBER,
	CRIME_DECILE NUMBER,
	EXCLUDE_CRIME_DOMAIN_FROM_CRIME_ANALYSIS BOOLEAN,
	HOUSING_BARRIERS_SCORE NUMBER,
	HOUSING_BARRIERS_RANK NUMBER,
	HOUSING_BARRIERS_DECILE NUMBER,
	LIVING_ENVIRONMENT_SCORE NUMBER,
	LIVING_ENVIRONMENT_RANK NUMBER,
	LIVING_ENVIRONMENT_DECILE NUMBER,
	TOTAL_POPULATION_MID_2022 NUMBER,
	DEPENDENT_CHILDREN_0_15_MID_2022 NUMBER,
	OLDER_POPULATION_60_PLUS_MID_2022 NUMBER,
	WORKING_AGE_POPULATION_MID_2022 NUMBER
);

-- Copying the cleaned English deprivation data from the Amazon s3 bucket to the table
COPY INTO DEPRIVATION_ENGLAND_CLEAN (
	LSOA_CODE,
	LSOA_NAME,
	LOCAL_AUTHORITY_CODE,
	LOCAL_AUTHORITY_NAME,
	IMD_SCORE,
	IMD_RANK,
	IMD_DECILE,
	INCOME_SCORE,
	INCOME_RANK,
	INCOME_DECILE,
	EMPLOYMENT_SCORE,
	EMPLOYMENT_RANK,
	EMPLOYMENT_DECILE,
	EDUCATION_SCORE,
	EDUCATION_RANK,
	EDUCATION_DECILE,
	HEALTH_SCORE,
	HEALTH_RANK,
	HEALTH_DECILE,
	CRIME_SCORE,
	CRIME_RANK,
	CRIME_DECILE,
	EXCLUDE_CRIME_DOMAIN_FROM_CRIME_ANALYSIS,
	HOUSING_BARRIERS_SCORE,
	HOUSING_BARRIERS_RANK,
	HOUSING_BARRIERS_DECILE,
	LIVING_ENVIRONMENT_SCORE,
	LIVING_ENVIRONMENT_RANK,
	LIVING_ENVIRONMENT_DECILE,
	TOTAL_POPULATION_MID_2022,
	DEPENDENT_CHILDREN_0_15_MID_2022,
	OLDER_POPULATION_60_PLUS_MID_2022,
	WORKING_AGE_POPULATION_MID_2022
)
FROM (
    SELECT 
        $1,                        
        $2,                        
        $3,
        $4,
        $5,                        
        $6,                        
        $7,
        $8,
        $9,                        
        $10,                        
        $11,
        $12,
        $13,                        
        $14,                        
        $15,
        $16,
        $17,                        
        $18,                        
        $19,
        $20,
        $21,                        
        $22,                        
        $23,
        $24,
        $25,                        
        $26,                        
        $27,
        $28,
        $29,                        
        $30,
        $31,
        $32,
        $33
    FROM @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/deprivation_england_clean.csv
)
FILE_FORMAT = (FORMAT_NAME = 'CLEAN_CSV_FORMAT')
ON_ERROR = 'ABORT_STATEMENT'
FORCE = TRUE;

### Step 2: Create Feature Transformation Function

In [ ]:
def featureTransformation(crime, population, walesDeprivation, englandDeprivation):
    
    # Copy of dataframes is created and used to ensure original dataframes are not overwritten    
    crime = crime.copy()
    population = population.copy()
    walesDeprivation = walesDeprivation.copy()
    englandDeprivation = englandDeprivation.copy()
    
    ## -----------------------------------------------------------------------
    ## Crime data feature transformations and additions 
    ## -----------------------------------------------------------------------
    
    crime['YEAR'] = crime['YEAR'].astype(int)
    crime['MONTH'] = crime['MONTH'].astype(int)
    
    # Clean Force names to be just the name of the police force without "Force" or other keywords
    crime['FORCE'] = crime['REPORTED_BY'].str.replace("Police", "").str.strip()
    crime.loc[crime['FORCE'].str.contains('Metropolitan', case=False), 'FORCE'] = 'Metropolitan'
    crime['FORCE'] = crime['FORCE'].str.title()

    # Standardize 'Crime type'
    crime['CRIME_TYPE'] = crime['CRIME_TYPE'].str.strip().str.title()

    ## Addition of Quarter column for quarterly analysis of crime
    monthToQuarter = {
        1: 1, 2: 1, 3: 1,
        4: 2, 5: 2, 6: 2,
        7: 3, 8: 3, 9: 3,
        10: 4, 11: 4, 12: 4
    }
    
    crime['QUARTER'] = crime['MONTH'].map(monthToQuarter)
    
    ## Addition of District column to broaden crime to general districts within forces rather than specific LDOAs 
    crime['DISTRICT'] = crime['LSOA_NAME'].str.rsplit(' ', n=1).str[0].str.strip()
    
    # Dropped Reported by column since a cleaner, standardised Force column was derived from it
    crime = crime.drop(columns=["FALLS_WITHIN", "REPORTED_BY"])
    
    ## ----------------------------------------------------
    ## Population estimation & feature transforming
    ## ----------------------------------------------------

    # Filters the population tables to include only the records with 2021 to 2024
    p2021 = population[population['YEAR'] == 2021].set_index('FORCE_NAME')['TOTAL_POPULATION']
    p2024 = population[population['YEAR'] == 2024].set_index('FORCE_NAME')['TOTAL_POPULATION']
    
    # Formula to calculate annual growth between 2021 to 2024
    annualGrowthRate = ((p2024/p2021) ** (( (1/3)))) - 1

    # Fetches the latest 2024 population for each force and the annual growth rate with the above formula
    futureRows = []
    for force in population['FORCE_NAME'].unique():
        popLatest = p2024[force]
        rate = annualGrowthRate[force]
        
        # Estimates 2025 & 2026
        pop2025 = round(popLatest * (1 + rate))
        pop2026 = round(pop2025 * (1 + rate))
        
        # Appends the rows to an array
        futureRows.append({'FORCE_NAME': force, 'YEAR': 2025, 'TOTAL_POPULATION': pop2025})
        futureRows.append({'FORCE_NAME': force, 'YEAR': 2026, 'TOTAL_POPULATION': pop2026})
    
    # Converts the array into a dataframe and concatenates it to the population table
    popEstimateDf = pd.DataFrame(futureRows)
    populationComplete = pd.concat([population, popEstimateDf], ignore_index=True)

    # Clean Force names to be just the name of the police force without "Force" or other keywords
    populationComplete['FORCE'] = populationComplete['FORCE_NAME'].str.replace("Police", "").str.strip()
    populationComplete.loc[populationComplete['FORCE'].str.contains('Metropolitan', case=False), 'FORCE'] = 'Metropolitan'
    populationComplete['FORCE'] = populationComplete['FORCE'].str.title()
    
    # Drops FORCE_NAME column since standardised FORCE column was created
    populationComplete = populationComplete.drop(columns=['FORCE_NAME'])
    
    # Join the Crime and population data by the Force and Year with a left join
    crimePopulation = pd.merge(crime, populationComplete, on=['FORCE', 'YEAR'], how='left')
    
    ## ----------------------------------------------------
    ## Deprivation cleaning and feature transforming
    ## ----------------------------------------------------
    
    # Maps "South Wales" force to Welsh LSOA deprivation records by checking active crime LSOA codes
    southWalesCrime = crimePopulation[crimePopulation['FORCE'] == 'South Wales']
    targetLsoas = southWalesCrime['LSOA_CODE'].unique()
    walesDeprivation['FORCE'] = 'South Wales'
    walesDeprivation['FORCE'] = walesDeprivation['FORCE'].where(
        walesDeprivation['LSOA_CODE'].isin(targetLsoas), 
        None
    )

    # Map of English forces boroughs to their respective police force 
    englandForceMap = {
        # Metropolitan
        'Barking and Dagenham': 'Metropolitan',
        'Barnet': 'Metropolitan',
        'Bexley': 'Metropolitan',
        'Brent': 'Metropolitan',
        'Bromley': 'Metropolitan',
        'Camden': 'Metropolitan',
        'Croydon': 'Metropolitan',
        'Ealing': 'Metropolitan',
        'Enfield': 'Metropolitan',
        'Greenwich': 'Metropolitan',
        'Hackney': 'Metropolitan',
        'Hammersmith and Fulham': 'Metropolitan',
        'Haringey': 'Metropolitan',
        'Harrow': 'Metropolitan',
        'Havering': 'Metropolitan',
        'Hillingdon': 'Metropolitan',
        'Hounslow': 'Metropolitan',
        'Islington': 'Metropolitan',
        'Kensington and Chelsea': 'Metropolitan',
        'Kingston upon Thames': 'Metropolitan',
        'Lambeth': 'Metropolitan',
        'Lewisham': 'Metropolitan',
        'Merton': 'Metropolitan',
        'Newham': 'Metropolitan',
        'Redbridge': 'Metropolitan',
        'Richmond upon Thames': 'Metropolitan',
        'Southwark': 'Metropolitan',
        'Sutton': 'Metropolitan',
        'Tower Hamlets': 'Metropolitan',
        'Waltham Forest': 'Metropolitan',
        'Wandsworth': 'Metropolitan',
        'Westminster': 'Metropolitan',
        'City of London': 'Metropolitan',
        # West Midlands
        'Birmingham': 'West Midlands',
        'Coventry': 'West Midlands',
        'Dudley': 'West Midlands',
        'Sandwell': 'West Midlands',
        'Solihull': 'West Midlands',
        'Walsall': 'West Midlands',
        'Wolverhampton': 'West Midlands',
        # Sussex
        'Adur': 'Sussex',
        'Arun': 'Sussex',
        'Brighton and Hove': 'Sussex',
        'Chichester': 'Sussex',
        'Crawley': 'Sussex',
        'Eastbourne': 'Sussex',
        'Hastings': 'Sussex',
        'Horsham': 'Sussex',
        'Lewes': 'Sussex',
        'Mid Sussex': 'Sussex',
        'Rother': 'Sussex',
        'Wealden': 'Sussex',
        'Worthing': 'Sussex',
    }
    
    # Uses above map to define the force for each LSOA
    englandDeprivation['FORCE'] = englandDeprivation['LOCAL_AUTHORITY_NAME'].map(englandForceMap)
    
    # Standardise English LSOA features
    englandDeprivation['DEPRIVATION_PERCENTILE'] = (englandDeprivation['IMD_RANK'] - 1) / (englandDeprivation['IMD_RANK'].max() - 1)
    englandDeprivation['IS_DECILE_1'] = np.where(englandDeprivation['IMD_DECILE'] == 1, 1, 0)
    englandDeprivation['INCOME_DECILE'] = englandDeprivation['INCOME_DECILE']

    # Standardise Welsh LSOA features
    walesDeprivation['DEPRIVATION_PERCENTILE'] = (walesDeprivation['WIMD_RANK'] - 1) / (walesDeprivation['WIMD_RANK'].max() - 1)
    walesDeprivation['IS_DECILE_1'] = np.where(walesDeprivation['WIMD_DECILE'] == 1, 1, 0)
    walesDeprivation['INCOME_DECILE'] = walesDeprivation['INCOME_DECILE']

    # Join the 2 deprivation tables together
    combined_iod = pd.concat([
        englandDeprivation[['LSOA_CODE', 'FORCE', 'DEPRIVATION_PERCENTILE', 'INCOME_DECILE', 'IS_DECILE_1']],
        walesDeprivation[['LSOA_CODE', 'FORCE', 'DEPRIVATION_PERCENTILE', 'INCOME_DECILE', 'IS_DECILE_1']]
    ], ignore_index=True)
    
    # Join the crime and population merged table with the joint deprivation tables
    combinedCleanDf = pd.merge(
    crimePopulation, 
    combined_iod[['LSOA_CODE', 'DEPRIVATION_PERCENTILE', 'INCOME_DECILE', 'IS_DECILE_1']], 
    on=['LSOA_CODE'], 
    how='left'
    )
    
    # Filter out of boundary districts logged by Metropolitan police
    valid_met_districts = list(englandForceMap.keys())
    combinedCleanDf = combinedCleanDf[
        (combinedCleanDf['FORCE'] != 'Metropolitan') | (combinedCleanDf['DISTRICT'].isin(valid_met_districts))
    ]
        
    return combinedCleanDf

### Step 3: Convert Cleaned Data Tables to Dataframes

In [ ]:
from snowflake.snowpark.context import get_active_session

# Use the notebook's built-in active session
session = get_active_session()

# Defined variables for the respective SQL queries needed to access the tables
crimeDataQuery = """
select * from CRIME_ETL_DB.CLEAN.CRIME_DATA_CLEAN
"""

populationQuery = """
select FORCE_NAME, YEAR, TOTAL_POPULATION from CRIME_ETL_DB.CLEAN.POPULATION_CLEAN
"""

walesDeprivation = """
select * from CRIME_ETL_DB.CLEAN.DEPRIVATION_WALES_CLEAN
"""

englandDeprivation = """
select * from CRIME_ETL_DB.CLEAN.DEPRIVATION_ENGLAND_CLEAN
"""

# Selecting the crime data with the respective SQL and coverting the output to a dataframe
crimeRun = session.sql(crimeDataQuery)
crimeDf = crimeRun.to_pandas()

# Selecting the population data with the respective SQL and coverting the output to a dataframe
populationRun = session.sql(populationQuery)
populationDf = populationRun.to_pandas()

# Selecting the Welsh deprivation data with the respective SQL and coverting the output to a dataframe
walesRun = session.sql(walesDeprivation)
walesDepDf = walesRun.to_pandas()

# Selecting the English deprivation data with the respective SQL and coverting the output to a dataframe
englandRun = session.sql(englandDeprivation)
englandDepDf = englandRun.to_pandas()

### Step 4: Apply Feature Transformation Function

In [ ]:
# Applying the feature transformation layer to the primary crime datasets as well as the enrichment deprivation and population data
combinedData = featureTransformation(crimeDf, populationDf, walesDepDf, englandDepDf)

## Aggregation & Export Layer 

### Step 1: Create Aggregation & Export Function

In [ ]:
# Aggregate Layer and export layer combined (FORCE X YEAR X QUARTER X DISTRICT X CRIME_TYPE)
def aggregationExport(combinedCleanDf):

    # Defined grain
    grain = ['FORCE', 'YEAR', 'QUARTER', 'DISTRICT', 'CRIME_TYPE']
    
    # The aggregation
    aggregatedDf = combinedCleanDf.groupby(grain).agg(
        # Counts number of crimes per the grain
        IncidentCount=('CRIME_ID', 'count'),
        
        # Records the total population per the grain
        TotalPopulation=('TOTAL_POPULATION', 'first'),
        
        # Records the number of LSOAs the grain spanned
        UniqueLSOAsSpanned=('LSOA_CODE', 'nunique'),
        
        # Deprivation metrics in regard to the grain
        AvgDeprivationPercentile=('DEPRIVATION_PERCENTILE', 'mean'),
        AvgIncomeDecile=('INCOME_DECILE', 'mean'),
    ).reset_index()

    # Crime rate per 1000 people
    aggregatedDf['CrimeRatePer1k'] = (
        aggregatedDf['IncidentCount'] / aggregatedDf['TotalPopulation']
    ) * 1000

    # Average deprivation percentile per the grain 
    aggregatedDf['AvgDeprivationPercentile'] = (aggregatedDf['AvgDeprivationPercentile'] * 100).round(2)
    
    # Standardising the aggregation metrics
    aggregatedDf['AvgIncomeDecile'] = aggregatedDf['AvgIncomeDecile'].round(2)
    aggregatedDf['CrimeRatePer1k'] = aggregatedDf['CrimeRatePer1k'].round(4)
    
    grainDuplicates = aggregatedDf.duplicated(subset=['FORCE', 'YEAR', 'QUARTER', 'DISTRICT', 'CRIME_TYPE']).sum()
    print(f"Validation - Duplicate records at reporting grain: {grainDuplicates}")

    # Check for number of null values in the data
    print("\nFinal Missing Value Summary:")
    print(aggregatedDf[['FORCE', 'YEAR', 'QUARTER', 'DISTRICT', 'CRIME_TYPE']].isnull().sum())

    # General summary of final dataset statistics
    print(f"\n--- Final Dataset Summary Statistics ---")
    print(f"Total Grain Rows: {len(aggregatedDf)}")
    print(f"Unique Police Forces Covered: {aggregatedDf['FORCE'].nunique()} ({aggregatedDf['FORCE'].unique()})")
    print(f"Temporal Window Span: {aggregatedDf['YEAR'].min()} to {aggregatedDf['YEAR'].max()}")
    
    aggregatedDf.to_csv('processedAndAggregatedCrimeData.csv', index=False)
    print("\nData Sucessfully converted into CSV file.")
    
    return aggregatedDf

### Step 2: Apply aggregationExport

In [ ]:
# Applying the combined aggregation and export layer to the combined data from the previous layer
final = aggregationExport(combinedData)